# Coordinate the human review

Use this notebook to check that the review files are ready, give each reviewer their notebook,
track progress, and summarize the results after both people finish. Ratings are entered in
[reviewer_1.ipynb](reviewer_1.ipynb) and [reviewer_2.ipynb](reviewer_2.ipynb).

The summary and note-saving options start off. A normal run checks the files and completion
counts without changing anyone's answers. If something is missing, it shows as pending. An error
means we need to check a file before continuing.

## Before sending files to the reviewers

Check that:

- Notebook 01 has reserved the same 100 articles for both methods. We have not used them for model-selection review or topic naming, although they were included in fitting.
- We have chosen the modern model and confirmed its topic labels.
- Laura has supplied the LDA results, confirmed its labels, and recorded that she used the same reserved sample.
- [The comparison notebook](../comparison/pipeline_comparison.ipynb) has created the review packet, its manifest/status files, and two blank answer CSVs.
- Two people have been assigned to reviewer 1 and reviewer 2. Each will review both labels for each article: 200 items per person.

If the packet is missing, finish the relevant steps in modern notebooks 01–03 and the comparison
notebook first. Start Jupyter with `uv run jupyter lab`. Keep the sample, model assignments, and
labels fixed while the review is in progress.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ROOT = REPO / "final-project/code/modern-methods"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from src.reviewer import load_session
from shared.human_review import summarize_reviews

## Step 1 — find the review folder

Leave `PACKET_FOLDER = None` to use the selected model's folder. If the team is using another
folder, enter its absolute path. It needs the common packet, the freeze/status files, and both
reviewers' answer CSVs.

The default location is:
`final-project/code/modern-methods/Results/comparison/<selected-run>/human_review/`.

In [2]:
# OPTIONAL: use an absolute path only if the team has a separate packet folder.
PACKET_FOLDER = None       # Example: Path("/absolute/path/to/human_review")
REVIEW_FOLDER = None
COMMON_FILES = ["review_packet.csv", "review_freeze.json", "review_status.json"]
REVIEWER_FILES = ["reviewer_1.csv", "reviewer_2.csv"]

if PACKET_FOLDER is not None:
    REVIEW_FOLDER = Path(PACKET_FOLDER).expanduser().resolve()
else:
    selection_file = ROOT / "Results/modern/selection.json"
    selection = json.loads(selection_file.read_text()) if selection_file.exists() else {}
    if selection.get("run_id"):
        REVIEW_FOLDER = ROOT / "Results/comparison" / selection["run_id"] / "human_review"

if REVIEW_FOLDER is None:
    print("Pending: finish model selection and labels, then generate the joint review packet in pipeline_comparison.ipynb.")
else:
    missing = [name for name in COMMON_FILES + REVIEWER_FILES if not (REVIEW_FOLDER / name).exists()]
    print("Packet folder:", REVIEW_FOLDER)
    if missing:
        print("Pending files:", ", ".join(missing))
        print("Use pipeline_comparison.ipynb to prepare the packet, or collect the missing original reviewer files.")
    else:
        print("All required files are present. Run Step 3 to validate them and check completion.")

Pending: finish model selection and labels, then generate the joint review packet in pipeline_comparison.ipynb.


## Step 2 — give each reviewer their files

| Person | Notebook | Answer file |
| --- | --- | --- |
| Reviewer 1 | [reviewer_1.ipynb](reviewer_1.ipynb) | `reviewer_1.csv` |
| Reviewer 2 | [reviewer_2.ipynb](reviewer_2.ipynb) | `reviewer_2.csv` |

Each notebook already has the correct reviewer ID. Give each person `review_packet.csv`,
`review_freeze.json`, `review_status.json`, and their own answer CSV. They do not need the other
reviewer's answers. Each person should use their own notebook copy and kernel.

Both notebooks use the same 1–5 fit scale, an article-relevance flag, a label-ambiguity flag,
and optional notes. Unassigned articles (topic −1) get flags and notes but no fit score.
Reviewers can see the model names, so the review is not blinded.

Ask each person to save a working copy as `reviewer_1_workbook.ipynb` or `reviewer_2_workbook.ipynb`
inside the packet folder, where Git ignores it. Collect the answer CSVs when they finish.
Discuss disagreements afterward, keeping the original independent ratings.

## Step 3 — check progress

This cell checks the answer files and shows completion counts without displaying ratings or notes.
An assigned item needs a fit score and both flags. An unassigned item needs both flags. Notes are
optional. Missing files show as pending.

Checking progress does not change answers. If a reviewer finds a problem with an article or label,
record it and investigate before changing any review inputs.

In [3]:
PROGRESS = None
if REVIEW_FOLDER is None or any(not (REVIEW_FOLDER / name).exists() for name in COMMON_FILES):
    print("Pending: locate a complete frozen packet in Step 1.")
else:
    progress_rows = []
    for reviewer in ["reviewer_1", "reviewer_2"]:
        if not (REVIEW_FOLDER / f"{reviewer}.csv").exists():
            progress_rows.append(dict(reviewer=reviewer, completed=None, total=None, remaining=None, status="file pending"))
            continue
        session = load_session(REVIEW_FOLDER, reviewer)
        total = len(session["packet"])
        completed = int(session["complete"].sum())
        progress_rows.append(dict(reviewer=reviewer, completed=completed, total=total,
                                  remaining=total-completed, status="complete" if completed == total else "in progress"))
    PROGRESS = pd.DataFrame(progress_rows)
    display(PROGRESS)
    print("No individual ratings are displayed or modified. Confirm both submissions before enabling Step 4.")

Pending: locate a complete frozen packet in Step 1.


## Step 4 — summarize the completed reviews

After both reviewers submit their files, set both switches below to `True`. The code checks the
files again. Missing answers stay pending; invalid values or changed article/label information
raise an error.

| Output | Meaning |
| --- | --- |
| `sample_articles`, `rated_articles` | Number sampled and number with numeric fit ratings for each model |
| `unassigned_articles` | Articles with topic −1, excluded from fit ratings |
| `mean_1`, `mean_2` | Each reviewer's average fit rating |
| `exact_agreement`, `within_one_agreement` | Share of paired ratings that match or differ by at most one point |
| `weighted_kappa` | Agreement beyond chance, with more weight given to large disagreements |
| `kappa_ci_low`, `kappa_ci_high` | 95% interval from 1,000 bootstrap resamples of articles, keeping ratings paired |
| `valid_bootstrap_samples`, `kappa_status` | Number of usable resamples and whether kappa could be calculated |

Agreement tells us how consistently reviewers scored, not whether a model is correct. Report
sample sizes with the scores. An undefined kappa is not the same as zero agreement.

The extra `outlier_review.csv` is for written observations and is not added to the 100-article
agreement sample. This step saves the summaries and keeps both original answer files. The merged
`review_examples.csv` contains individual ratings and stays local; read it only after both
reviewers finish.

In [4]:
# COORDINATOR ONLY: leave both False until both reviewers have submitted independently.
BOTH_REVIEWERS_FINISHED = False
RUN_TEAM_SUMMARY = False
TEAM_SUMMARY = None

if BOTH_REVIEWERS_FINISHED and RUN_TEAM_SUMMARY:
    if REVIEW_FOLDER is None:
        print("Pending: choose the frozen packet folder in Step 1.")
    else:
        missing = [name for name in COMMON_FILES + REVIEWER_FILES if not (REVIEW_FOLDER / name).exists()]
        if missing:
            print("Pending files:", ", ".join(missing))
        else:
            TEAM_SUMMARY = summarize_reviews(REVIEW_FOLDER, bootstrap_samples=1000, seed=42)
            display(TEAM_SUMMARY)
            if isinstance(TEAM_SUMMARY, pd.DataFrame):
                print("Saved human_evaluation.csv and rating_distributions.csv in", REVIEW_FOLDER)
                print("Original reviewer files are unchanged. Continue to the interpretation and handoff steps.")
else:
    print("Team summary not run. Independent reviewer scores have not been compared.")

Team summary not run. Independent reviewer scores have not been compared.


## Step 5 — write our interpretation

After checking the summary, fill in the notes below:

- How well did the labels describe the articles?
- Where did reviewers disagree? Were the labels unclear, or did the articles cover several topics?
- What did reviewers notice about unassigned or irrelevant articles? Use IDs and brief paraphrases.
- How do the small sample, two team reviewers, and visible model names limit our conclusions?

Remember that the sample was set aside from topic naming, not model fitting. Save the notes as
`team_interpretation.csv` when ready. This leaves the original ratings and manuscript unchanged.

In [5]:
# Replace the empty strings with your team's interpretation of the completed results.
TEAM_NOTES = {
    "article_topic_fit": "",
    "reviewer_disagreements": "",
    "unassigned_or_irrelevant_examples": "",
    "limitations_and_report_wording": "",
}
SAVE_TEAM_NOTES = False

if SAVE_TEAM_NOTES:
    if not isinstance(TEAM_SUMMARY, pd.DataFrame):
        raise ValueError("Run a complete team summary before saving interpretation notes.")
    if any(not text.strip() for text in TEAM_NOTES.values()):
        raise ValueError("Fill in each interpretation field, or state that no conclusion is supported.")
    notes_table = pd.DataFrame(TEAM_NOTES.items(), columns=["section", "human_interpretation"])
    notes_table.to_csv(REVIEW_FOLDER / "team_interpretation.csv", index=False)
    display(notes_table)
else:
    print("No interpretation notes saved. Fill these in after both reviewers finish.")

No interpretation notes saved. Fill these in after both reviewers finish.


## Step 6 — use the results in PR3

Keep the packet and both original answer CSVs. Use `human_evaluation.csv`,
`rating_distributions.csv`, and the team's notes in the report, including sample sizes, outlier
coverage, and uncertainty. Rerun [pipeline_comparison.ipynb](../comparison/pipeline_comparison.ipynb)
to update the combined results and checklist. Edit the manuscript separately.

Keep full article text and individual answers local. Before sharing this notebook as a blank
copy, reset any typed notes and save switches, then clear the outputs.